# TrainLM T1: TorchTPU single-device probe

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dhiraj309/TrainLM/blob/docs/01-t1-v5e1-probe/notebooks/TorchTPU_T1_v5e1_Probe.ipynb)

Open this notebook in Colab to run or copy its code cells directly, including on mobile. Choose a hosted runtime through Runtime → Change runtime type, or use Connect → Connect to a local runtime for your configured Cloud TPU VM. The notebook link does not provision hardware.

This notebook runs device detection and the probe in one Python process. It checks package provenance, forward execution, autograd, an AdamW update, and optionally torch.compile.

Hosted Kaggle currently documents TPU v5e-8, not v5e-1. Colab's public TPU notebook documents TPU v2. Neither hosted accelerator is v5e-1 evidence. For the planned v5e-1 check, use a one-chip Cloud TPU v5e VM (Google lists machine type ct5lp-hightpu-1t) with the approved TorchTPU runtime; a Colab frontend connected to that VM runs on the VM's hardware.

This notebook does not install packages or treat a generic accelerator as TorchTPU. A TPU pass is labeled with the runtime platform and must be paired with the VM/accelerator identifier to establish the hardware generation. Use v5e-8 for checks that require multiple devices.

The setup cell creates a fresh source snapshot and prints its exact commit. The last cell downloads a compact evidence report, so mobile users do not need to select or copy logs.

In [ ]:
from pathlib import Path
import json
import os
import runpy
import subprocess
import sys
import tempfile

PROBE_BRANCH = "docs/01-t1-v5e1-probe"
REPOSITORY_URL = "https://github.com/Dhiraj309/TrainLM.git"
# Each setup run gets a fresh snapshot; existing checkouts are preserved.
session_dir = Path(tempfile.mkdtemp(prefix="trainlm-t1-"))
repo_root = session_dir / "TrainLM"
subprocess.run(
    ["git", "clone", "--depth", "1", "--single-branch", "--branch", PROBE_BRANCH,
     REPOSITORY_URL, str(repo_root)], check=True, timeout=120,
)
SOURCE_REVISION = subprocess.check_output(
    ["git", "-C", str(repo_root), "rev-parse", "HEAD"], text=True,
).strip()
subprocess.run(["git", "-C", str(repo_root), "checkout", "--detach", SOURCE_REVISION],
               check=True, timeout=30)


def assert_source():
    revision = subprocess.check_output(
        ["git", "-C", str(repo_root), "rev-parse", "HEAD"], text=True,
    ).strip()
    changes = subprocess.check_output(
        ["git", "-C", str(repo_root), "status", "--porcelain"], text=True,
    ).strip()
    if revision != SOURCE_REVISION or changes:
        raise RuntimeError("Probe source changed. Rerun setup to start a new evidence session.")


assert_source()
RUN_PROCESS = runpy.run_path(str(repo_root / "src/trainlm/runtime/torchtpu_probe_process.py"))["run_probe_process"]
EVIDENCE = runpy.run_path(str(repo_root / "src/trainlm/runtime/torchtpu_evidence.py"))
T1_RESULTS = dict.fromkeys(("package_process", "eager_process", "compile_process",
                          "lifecycle_report", "distributed_process"))
NOTEBOOK_HOST = ("Colab" if "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules
                 else "Kaggle" if "KAGGLE_KERNEL_RUN_TYPE" in os.environ or "KAGGLE_URL_BASE" in os.environ
                 else "other")


def run_script(script, arguments=(), *, timeout=300, shutdown_grace=5):
    assert_source()
    result = RUN_PROCESS([sys.executable, str(repo_root / script), *arguments],
                         cwd=repo_root, timeout=timeout, shutdown_grace=shutdown_grace)
    assert_source()
    return result


print(f"notebook_host={NOTEBOOK_HOST}")
print(f"source_revision={SOURCE_REVISION}")
print("Fresh source snapshot ready; previous notebook results cleared.")

## Read-only package preflight

This cell compares installed metadata with TrainLM's source requirements and TorchTPU's declared requirements, then runs `pip check`. It does not import torch, initialize a TPU, or install packages. A mismatch is useful T1 evidence; the standalone smoke can still run for diagnosis, but it does not establish a supported TrainLM installation.


In [ ]:
T1_RESULTS["package_process"] = None
T1_RESULTS["package_process"] = run_script(
    "examples/torchtpu_t1_install_preflight.py", timeout=60,
)
print(json.dumps(EVIDENCE["profile_evidence"](T1_RESULTS["package_process"]), indent=2))

## Try the one-command doctor

This invokes the same `trainlm doctor` command from the fresh source checkout without installing TrainLM into the Colab runtime. It performs TPU discovery, so expect one additional short TPU initialization. An incompatible package result is reported for review and does not stop the eager smoke.


In [ ]:
assert_source()
doctor_code = (
    "import sys; "
    "sys.path.insert(0, '" + str(repo_root / 'src') + "'); "
    "from trainlm.cli import main; raise SystemExit(main([\'doctor\']))"
)
doctor_result = RUN_PROCESS([sys.executable, "-c", doctor_code],
                            cwd=repo_root, timeout=300)
assert_source()
if doctor_result["stdout"]:
    print(doctor_result["stdout"], end="")
if doctor_result["stderr"]:
    print("doctor_stderr_present=true")
doctor_payload = json.loads(doctor_result["stdout"].splitlines()[-1])
print(f"doctor_command_exit={doctor_result['exit_code']}")
print(f"doctor_report_status={doctor_payload['status']}")


## Detect runtime and device

The probe uses the official TorchTPU device API and verifies a real tensor allocation inside the probe process. The standard hosted options are documented as Colab TPU v2 and Kaggle v5e-8. A Colab connection to a Cloud TPU VM uses that VM's hardware.

In [ ]:
DEVICE = "auto"  # Or explicitly "tpu" or "cpu"; CPU results remain labeled.
print(f"selected_device={DEVICE} (resolved inside the probe process)")
print("Hardware comes from the attached runtime, not the notebook host label.")

## Eager single-device smoke

The probe checks finite loss and gradients plus an actual optimizer update. Device detection and training share one child process. Execution is limited to 300 seconds; timeout or interruption terminates the whole child group. Fatal runtime shutdown output fails the check even if its exit code is zero.

The normalized result is retained for the download cell even when this cell raises an error. Raw child output remains in this notebook for local diagnosis and is excluded from the shared report.

In [ ]:
def run_probe(*, compiled=False):
    arguments = ["--device", DEVICE]
    if DEVICE == "cpu":
        arguments.append("--allow-cpu")
    if compiled:
        arguments.append("--compile")
    result = run_script("examples/torchtpu_t1_v5e1_probe.py", arguments,
                        timeout=900 if compiled else 300)
    if result["stdout"]:
        print(result["stdout"], end="")
    if result["stderr"]:
        print(result["stderr"], file=sys.stderr, end="")
    return result


T1_RESULTS["eager_process"] = None
T1_RESULTS["eager_process"] = run_probe()
eager = EVIDENCE["smoke_evidence"](T1_RESULTS["eager_process"])
print(json.dumps(eager, indent=2))
if eager["status"] != "passed":
    raise RuntimeError("Eager check failed; download the evidence report below before investigating.")

## Optional compile smoke

Run this separately because compilation can take longer and may expose a runtime/compiler issue. If it fails, keep the error as evidence; the script does not silently fall back to eager execution.

In [ ]:
RUN_COMPILE = False  # Enable when ready for explicit torch.compile validation.
T1_RESULTS["compile_process"] = None
if RUN_COMPILE:
    T1_RESULTS["compile_process"] = run_probe(compiled=True)
    compiled = EVIDENCE["smoke_evidence"](T1_RESULTS["compile_process"], compiled=True)
    print(json.dumps(compiled, indent=2))
    if compiled["status"] != "passed":
        raise RuntimeError("Compile check failed; the download cell retains this result.")
else:
    print("compile=not_run; set RUN_COMPILE = True when ready")

## Optional process lifecycle check

This check is disabled by default. On an isolated single-device TPU runtime, it starts two sequential fresh probe processes, verifies each eager smoke and device report, and records whether the observations agree. Each child has a 300-second timeout and its process group is terminated on timeout or interruption. A failure is lifecycle evidence to investigate, not a reason to retry automatically. Run it only when ready for hardware validation.


In [ ]:
RUN_LIFECYCLE = False
T1_RESULTS["lifecycle_report"] = None
if RUN_LIFECYCLE:
    assert_source()
    run_lifecycle = runpy.run_path(str(repo_root / "examples/torchtpu_t1_lifecycle_probe.py"))["run_lifecycle"]
    T1_RESULTS["lifecycle_report"] = run_lifecycle(session_dir / "lifecycle", repeats=2, timeout=300)
    assert_source()
    lifecycle = EVIDENCE["lifecycle_evidence"](T1_RESULTS["lifecycle_report"])
    print(json.dumps(lifecycle, indent=2))
    if lifecycle["status"] != "passed":
        raise RuntimeError("Lifecycle check failed; the download cell retains this result.")
else:
    print("lifecycle=not_run; set RUN_LIFECYCLE = True on an isolated TPU runtime")

## Review the hardware scope

Record the actual accelerator and runtime/VM image beside the downloaded report. The notebook host name alone cannot establish v5e-1 evidence. This synthetic single-device check does not establish collective, sharding, gradient-parity, checkpoint, or performance behavior. The v5e-8 distributed notebook is a separate feasibility check.

## Download a shareable result

Run this cell even after a failed check. Colab offers a JSON download; other notebook hosts display a file link. The report includes the exact source revision, package comparison, normalized hardware observations and outcomes. Skipped checks stay `not_run`, and CPU results stay labeled as non-TPU evidence. Raw logs, environment variables, coordinator addresses and local artifact paths are excluded.

Share this JSON together with the selected runtime image and actual accelerator identity. This report requires owner review and does not certify native training, gradient-correct data parallelism, or performance. Rerun setup to clear all results and use a fresh source snapshot.

In [ ]:
assert_source()
report = EVIDENCE["build_evidence"](
    source_revision=SOURCE_REVISION, notebook_host=NOTEBOOK_HOST, **T1_RESULTS,
)
EVIDENCE_PATH = Path.cwd() / "trainlm_t1_evidence.json"
EVIDENCE_PATH.write_text(json.dumps(report, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(json.dumps({name: check["status"] for name, check in report["checks"].items()}, indent=2))
try:
    from google.colab import files
except ImportError:
    from IPython.display import FileLink, display
    display(FileLink(EVIDENCE_PATH.name))
else:
    files.download(str(EVIDENCE_PATH))